In [382]:
import pandas as pd
import numpy as np
import scipy.stats as stats

**Initial data cleanup**

Rounding to two decimal places and check for missing values in _general_data_cleanup.py_

**Validation**

In [383]:
data = pd.read_csv("Data/USO/USO_clean.csv")
data

,timestamp,open,high,low,close,volume
0,2006-04-10 13:00:00+00:00,546.00,548.00,545.28,545.60,50862.50
1,2006-04-10 14:00:00+00:00,545.60,546.48,542.00,542.64,149875.00
2,2006-04-10 15:00:00+00:00,542.48,544.80,542.24,543.12,85625.00
3,2006-04-10 16:00:00+00:00,543.28,543.84,541.36,543.20,99600.00
4,2006-04-10 17:00:00+00:00,543.12,544.80,542.16,542.72,46700.00
...,...,...,...,...,...,...
72224,2026-10-05 19:00:00+00:00,143.45,144.02,143.41,144.01,388765.10
72225,2026-10-05 20:00:00+00:00,143.99,144.25,143.67,144.04,73482.56
72226,2026-10-05 21:00:00+00:00,144.07,144.20,143.58,144.06,39440.34
72227,2026-10-05 22:00:00+00:00,144.07,144.20,143.57,143.98,9008.82


High/low validation

In [384]:
sum(data["high"] < data["low"])

0

Chronology validation

In [385]:
sum(data["timestamp"] != data["timestamp"].sort_values())

0

Time jumps

In [386]:
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["time_diffs"] = data["timestamp"] - data["timestamp"].shift(1)

print(time_diffs.head())

data["time_diffs"] = data["time_diffs"].dt.total_seconds() / 3600

time_diffs = data["time_diffs"]
time_diffs[0] = 0

print(np.sort(time_diffs.unique()))

0    0.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: time_diffs, dtype: float64
[  0.   1.   2.   3.   4.   8.   9.  10.  11.  12.  13.  14.  15.  16.
  17.  18.  33.  34.  35.  36.  37.  38.  39.  40.  41.  42.  43.  44.
  56.  57.  58.  59.  60.  61.  62.  63.  64.  65.  66.  67.  68.  81.
  82.  83.  84.  85.  86.  87.  88.  89. 109. 114.]


Time elapsed clusters dictionary

In [387]:
time_elapsed_clusters = {
    **{i: "consecutive" for i in range(1,5)},
    **{i: "intraday" for i in range(8,19)},
    **{i: "one_day" for i in range(33,45)},
    **{i: "weekend" for i in range(56,69)},
    **{i: "long_break" for i in range(81,89)},
    109: "long_break",
    114: "long_break"
}

Range of data in years:

In [388]:

(data["timestamp"][len(data["timestamp"])-1] - data["timestamp"][0]).days/365

20.5013698630137

In [389]:
print(data.head())

                  timestamp    open    high     low   close    volume  \
0 2006-04-10 13:00:00+00:00  546.00  548.00  545.28  545.60   50862.5   
1 2006-04-10 14:00:00+00:00  545.60  546.48  542.00  542.64  149875.0   
2 2006-04-10 15:00:00+00:00  542.48  544.80  542.24  543.12   85625.0   
3 2006-04-10 16:00:00+00:00  543.28  543.84  541.36  543.20   99600.0   
4 2006-04-10 17:00:00+00:00  543.12  544.80  542.16  542.72   46700.0   

   time_diffs  
0         NaN  
1         1.0  
2         1.0  
3         1.0  
4         1.0  


Vector of shifts between close and open prices for consecutive hours

In [390]:
data_shifted = data
data_shifted["open"] = data_shifted["open"].shift(-1)
shifts = data_shifted["open"] - data_shifted["close"]

print(shifts.head())

shifts /= data_shifted["close"]

print(shifts.head())

average_shift = shifts.abs().mean()

print("\nAverage proportional shift: ", average_shift)

print("Max proportional shift: ", shifts.abs().max())

0    0.00
1   -0.16
2    0.16
3   -0.08
4    0.08
dtype: float64
0    0.000000
1   -0.000295
2    0.000295
3   -0.000147
4    0.000147
dtype: float64

Average proportional shift:  0.0011819566606780967
Max proportional shift:  0.20689655172413782


In [391]:
long_shifts = shifts[time_diffs > 4]
long_shifts.abs().max()

np.float64(0.029790890862217312)

In [399]:
big_shifts = shifts[shifts.abs() > 0.01]
len(big_shifts)

1113

In [400]:
time_elapsed_big_shifts = time_diffs[shifts.abs() > 0.01]

time_elapsed_small_shifts = time_diffs[shifts.abs() <= 0.01]

percentage_inclusion_in_big_shifts = (time_elapsed_big_shifts.value_counts()*100/time_elapsed_big_shifts.count()).sort_index()

time_diffs_percentage_inclusion = (time_elapsed_small_shifts.value_counts()*100/time_elapsed_small_shifts.count()).sort_index()

print("Rest of the sample:\n", time_diffs_percentage_inclusion.groupby(time_elapsed_clusters).sum())
print("Diff > 1%:\n", percentage_inclusion_in_big_shifts.groupby(time_elapsed_clusters).sum())

#time_diffs_percentage_inclusion = (time_diffs.value_counts()*100/time_diffs.count()).sort_index()

Rest of the sample:
 time_diffs
consecutive    92.855234
intraday        5.595163
long_break      0.187021
one_day         0.070309
weekend         1.286648
Name: count, dtype: float64
Diff > 1%:
 time_diffs
consecutive    93.441150
intraday        4.851752
long_break      0.269542
one_day         0.089847
weekend         1.347709
Name: count, dtype: float64


In [394]:
time_elapsed_small_shifts.value_counts().groupby(time_elapsed_clusters).sum()

time_diffs
consecutive    50945
intraday        1746
long_break        58
one_day           26
weekend          383
Name: count, dtype: int64

In [395]:
time_elapsed_big_shifts.value_counts().groupby(time_elapsed_clusters).sum()

time_diffs
consecutive    16129
intraday        2287
long_break        78
one_day           25
weekend          547
Name: count, dtype: int64

In [396]:
stats.ks_2samp(time_elapsed_big_shifts, time_elapsed_small_shifts, alternative='two-sided', mode="auto")

KstestResult(statistic=np.float64(0.15229665865169362), pvalue=np.float64(1.1285465705196585e-284), statistic_location=np.float64(1.0), statistic_sign=np.int8(-1))

In [397]:
stats.ttest_rel(time_elapsed_small_shifts.value_counts().groupby(time_elapsed_clusters).sum()
                , time_elapsed_big_shifts.value_counts().groupby(time_elapsed_clusters).sum(), alternative='two-sided')

TtestResult(statistic=np.float64(0.9740467359191124), pvalue=np.float64(0.38518842987639645), df=np.int64(4))